In [1]:
!pip install -q sentence-transformers faiss-cpu groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 5.4 MB/s eta 0:00:00


In [2]:
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from groq import Groq

In [3]:
client = Groq(
    api_key="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"
)

In [4]:
doc=["Large Language Models (LLMs) are AI systems trained on vast amounts of text data to understand and generate human-like language.LLMs can perform tasks such as text generation, summarization, translation, question answering, and code generation.Popular LLMs include models developed by OpenAI, Google DeepMind, and Meta AI.These models help automate communication, improve productivity, and enhance user experiences across various applications.",
     "Retrieval-Augmented Generation (RAG) combines information retrieval with LLMs to provide more accurate and up-to-date responses.Instead of relying only on its training data, a RAG system retrieves relevant information from external knowledge sources before generating an answer.RAG helps reduce hallucinations and improves the reliability of AI-generated content.It is widely used in chatbots, enterprise knowledge systems, document search applications, and customer support solutions.",
     "Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed. ML algorithms improve their performance over time by analyzing historical data and identifying relationships within it. It is widely used in applications such as recommendation systems, spam detection, fraud detection, predictive analytics, and customer segmentation.",
     "Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns from large amounts of data. Unlike traditional ML models, Deep Learning can automatically extract features from raw data, making it highly effective for tasks such as image recognition, speech processing, natural language understanding, and generative AI. Advances in computing power and large-scale datasets have made Deep Learning a key technology behind modern AI applications."]

In [5]:
model = SentenceTransformer('all-MiniLM-L6-v2')
doc_embeddings=model.encode(doc).astype('float32')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [6]:
doc_embeddings.shape

(4, 384)

In [7]:
dimension = doc_embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(doc_embeddings)

In [8]:
query="How does Machine Learning differ from traditional programming?"
query_embedding = model.encode([query]).astype("float32")

top_k = 2
distances, indices = index.search(query_embedding, top_k)

retrieved_chunks = [doc[i] for i in indices[0]]
print("Retrieved_chunks:")
for chunk in retrieved_chunks:
  print("-", chunk)


Retrieved_chunks:
- Machine Learning is a branch of Artificial Intelligence that enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed. ML algorithms improve their performance over time by analyzing historical data and identifying relationships within it. It is widely used in applications such as recommendation systems, spam detection, fraud detection, predictive analytics, and customer segmentation.
- Deep Learning is a specialized subset of Machine Learning that uses artificial neural networks with multiple layers to learn complex patterns from large amounts of data. Unlike traditional ML models, Deep Learning can automatically extract features from raw data, making it highly effective for tasks such as image recognition, speech processing, natural language understanding, and generative AI. Advances in computing power and large-scale datasets have made Deep Learning a key technology behind modern AI applications.


In [9]:
context = "\n\n".join(retrieved_chunks)

prompt = f"""
Answer the question using only the context below.

Context:
{context}

Question:
{query}

Answer:
"""
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.2,
    max_tokens=300
)

print("\nFinal Answer:\n")
print(response.choices[0].message.content)


Final Answer:

Machine Learning differs from traditional programming in that it enables computers to learn patterns from data and make predictions or decisions without being explicitly programmed. In traditional programming, computers are given a set of rules and instructions to follow, whereas in Machine Learning, computers improve their performance over time by analyzing historical data and identifying relationships within it.
